In [1]:
import numpy as np
import scipy.sparse as sp

from math import sqrt

import igl
import pyvista as pv
pv.set_jupyter_backend('trame')

In [2]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

In [3]:
v, f = igl.read_triangle_mesh("cat.off")

In [4]:
mesh = to_pyvista_mesh(v, f)
mesh.plot(show_edges=True)

Widget(value='<iframe src="http://localhost:53756/index.html?ui=P_0x1061fec70_0&reconnect=auto" class="pyvista…

In [5]:
# each idx corresponds to a face, each element is a component, highest component count = biggest component\
# Filter out noise components
(num_components, component_ids) = igl.facet_components(f)
unique, counts = np.unique(component_ids, return_counts=True)
count_per_component = dict(zip(unique, counts))

comp_vals = list(count_per_component.values())
largest_comp_idx = comp_vals.index(max(comp_vals))
filtered_f_idx = [i for i, comp in enumerate(component_ids) if comp == largest_comp_idx]

filtered_f = f[filtered_f_idx, :]

to_pyvista_mesh(v, filtered_f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:53756/index.html?ui=P_0x17cc725b0_1&reconnect=auto" class="pyvista…

In [6]:
vf, ni = igl.vertex_triangle_adjacency(f, len(v))
areas = igl.doublearea(v, f)
angles = igl.internal_angles(v, f)

A_flat = []
L_c = np.zeros((len(v), len(v)))
for i in range(len(v)):
    adj_faces = []
    # Find all incident faces for current vertex
    for j in range(0, ni[i+1] - ni[i]):
        cur_face = vf[ni[i] + j]
        adj_faces.append(cur_face)
        
        # Find a_ij and b_ij for i
        assoc_tri_verts = f[cur_face]
        angle_idx = np.where(assoc_tri_verts != i)[0]
        
        L_c[i, assoc_tri_verts[angle_idx[0]]] = angles[i, angle_idx[0]]
        L_c[i, assoc_tri_verts[angle_idx[1]]] = angles[i, angle_idx[1]]
        
    A_i = 1/3 * sum(areas[adj_faces])
    
    A_flat.append(A_i)
assert(len(A_flat) == len(v))
A = np.diag(A_flat)

In [10]:
t = igl.avg_edge_length(v, f) ** 2 # Paper gives t = h^2 as a good heuristic
dirac_delta = np.zeros(len(v))
dirac_delta[0] = 1 # Initialize starting point to vertex zero

# TODO: Convert to sparse mat, takes a long time to solve for larger meshes
u = np.linalg.solve(A - t * L_c, dirac_delta)

assert(len(u) == len(v))

366
366
